# Phase 7 — RAFT Training-Data Generation (self-distillation)

Builds the SFT dataset to fine-tune Qwen3 for **safety self-containment + prompt
compression** (see `LLM_PLAN.md` Phase 7, revised after Phase 6).

- **All targets (as run): base Qwen3-4B + RAG + safety-validator (self-distillation).**
- A **Gemini teacher backend is implemented and pluggable** (`teacher_share > 0`) for
  harder/emergency/diverse cases, but was **not used** in the final dataset — free-tier
  quota was unreliable and the self-distilled set passed the safety gate at 1.00.
- Every gold answer must pass `scan_unsafe`; targets are paired with a **compact**
  prompt; Unicode is normalized.

**Outputs:** `train.jsonl`, `val.jsonl`, `data_card.md`.

> **Final run (2026-07-26):** `n=600`, `teacher_share=0.0` → 600/600 kept
> (gate pass rate 1.00), train 540 / val 60, class mix 200/200/200.
> The committed dataset is `ckd_risk_app/phase7_data/`.

> **GPU runtime required** (for self-distillation).
> Only synthetic, de-identified profiles are ever sent to an external teacher.

## 1. Install dependencies

In [ ]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu google-genai textstat

## 2. Make the project importable (upload zip if needed)

In [ ]:
import os, sys, zipfile
from pathlib import Path
CANDIDATES = [
    '/content/ckd_risk_app_LLM_READY_INTERPRETABLE_FRONTEND',
    '/content/drive/MyDrive/ckd_risk_app_LLM_READY_INTERPRETABLE_FRONTEND',
    os.getcwd(),
]
def _has(p): return (Path(p)/'ckd_risk_app'/'llm').exists()
PROJECT_ROOT = next((p for p in CANDIDATES if _has(p)), None)
if PROJECT_ROOT is None:
    from google.colab import files
    print('Upload a .zip of the project (containing ckd_risk_app/).')
    up = files.upload()
    for name in up:
        if name.endswith('.zip'):
            zipfile.ZipFile(name).extractall('/content/project')
    for root,_d,_f in os.walk('/content/project'):
        if _has(root): PROJECT_ROOT = root; break
assert PROJECT_ROOT, 'ckd_risk_app/ not found.'
sys.path.insert(0, PROJECT_ROOT); os.chdir(PROJECT_ROOT)
import ckd_risk_app.llm
print('PROJECT_ROOT =', PROJECT_ROOT)

## 3. Build the BGE-M3 + FAISS index (idempotent)

In [ ]:
from ckd_risk_app.llm.rag.embed_index import build
build(embedder_spec='auto')   # BGE-M3 + FAISS

## 4. Gemini API key (free tier)

Get a free key at https://aistudio.google.com/apikey . Only synthetic profiles are sent.

In [ ]:
from getpass import getpass
import os
os.environ['GEMINI_API_KEY'] = getpass('GEMINI_API_KEY: ')

## 5. Generate the dataset

`n` = number of training profiles; `teacher_share` = fraction routed to Gemini.

**The committed dataset was generated with `teacher_share=0.0` (100% self-distillation).**
Set it above 0 only if you want to re-run the diversity experiment with an external
teacher; the Gemini key cell above is then required. Tip: try `n=60` first to
sanity-check, then scale to 600. Self-distillation runs on the GPU (~40–80 min on a T4).

In [ ]:
from ckd_risk_app.llm.data.generate_dataset import generate
result = generate(
    n=600,
    teacher_share=0.0,                 # AS RUN: 100% self-distillation, no external teacher
    self_model='qwen',                 # base Qwen3-4B for self-distillation
    gemini_model='gemini-2.5-flash',   # only used if teacher_share > 0
    out_dir='/content/phase7_data',
)

## 6. Review the data card

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('/content/phase7_data/data_card.md', encoding='utf-8').read()))

## 7. Inspect a few training examples

In [ ]:
import json
rows = [json.loads(l) for l in open('/content/phase7_data/train.jsonl', encoding='utf-8')]
print('train examples:', len(rows))
for ex in rows[:2]:
    print('='*70)
    print(ex['profile_id'], ex['risk_class'], '| teacher:', ex['teacher'])
    print('-'*70, '
[ASSISTANT TARGET]
', ex['messages'][2]['content'][:700])

## 8. Download the dataset

In [ ]:
from google.colab import files
files.download('/content/phase7_data/train.jsonl')
files.download('/content/phase7_data/val.jsonl')
files.download('/content/phase7_data/data_card.md')